# Nepali Summarization LoRA Fine-tune v6 — Llama 3.2 3B Instruct, Unsloth, Kaggle
### High-quality-synthetic version — zero-duplication mixing, no more mid-sentence cutoffs

The v5 run fixed the *overfitting on low-quality synthetic data* problem (dedup + 85/15 real-heavy
mix + real-data validation + early stopping). This version keeps every one of those safeguards and
adds three more, driven by what changed in your data and what you asked for this time:

1. **New synthetic source**: swaps the old 5k low-quality synthetic set for
   `iwasbinod/nep_summarization_1k_data-highquality` (small, curated, high quality).
2. **Guaranteed zero duplication.** With only ~1k high-quality rows, blindly asking for "15% of
   the real-data pool" can silently *upsample the same examples multiple times per epoch* —
   exactly the kind of repetition that causes memorization. The mixing function below now **caps
   each source's quota at its own unique size** and redistributes any shortfall to the source(s)
   that still have headroom, so no row is ever seen twice per epoch from either source.
3. **Fixes mid-sentence truncation at generation time.** `max_new_tokens` for summary generation
   is raised from 200 → 320 (Devanagari tends to tokenize into more pieces than English for the
   same semantic content, so 200 was cutting summaries off before their final sentence — this is
   the same failure mode that got LLaMA/Mistral summarization deprecated in your other runs).
4. Everything from v5 is retained: real-data-dominant 85/15 mix, in-training validation on
   held-out **real** XLSum-Nepali data, early stopping on real-data eval loss, and a
   base-vs-fine-tuned comparison on **both** real and synthetic eval sets plus the factual-grounding
   (`source_copy_jaccard`) regression check.
5. **New adapter name** so this never collides with any earlier (overfit) adapter:
   `iwasbinod/summarization-llama3.2-3b-real-syn-highquality`
   (Hugging Face repo IDs don't allow `+`, so the requested
   `real+syn-highquality` naming is rendered as `real-syn-highquality`.)

**On "always better than base":** no training run can be *guaranteed* to beat the base model —
that depends on your data and a bit of luck with the random seed. What this notebook guarantees
instead is that you'll **know** whether it did, on real data specifically, before you ever push
anything: `load_best_model_at_end` + `EarlyStoppingCallback` always keep the checkpoint with the
lowest real-data validation loss, and the final report explicitly deltas real-XLSum composite
score and factual grounding, base vs. fine-tuned, with an explicit warning if either regresses.
Read that report before you trust the push cell.

---

## 0. GPU check

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU DETECTED - enable a GPU accelerator in the notebook settings panel."


Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [2]:
import os
# Optimizes memory allocation for CUDA before any PyTorch initialization
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

print("Environment configured.")


Environment configured.


## 1. Install dependencies

In [3]:
# Install Unsloth and modern SFT training libraries
!pip install -q --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -q -U unsloth_zoo
# !pip install -q trl peft accelerate bitsandbytes datasets huggingface_hub rouge_score evaluate sacrebleu bert-score
!pip install -q trl peft accelerate bitsandbytes "datasets<3.0.0" huggingface_hub rouge_score evaluate sacrebleu bert-score

import torch
import time
import re
import gc
import math
import random
import numpy as np
from datasets import load_dataset, DatasetDict, concatenate_datasets
from huggingface_hub import login
from unsloth import FastLanguageModel

import evaluate as hf_evaluate
import sacrebleu
from bert_score import score as bert_score

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Connected GPU:", torch.cuda.get_device_name(0))


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 364.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 299.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 316.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 269.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 372.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 350.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 273.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 383.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 380.2 MB/s 

## 2. Load Nepali-only XLSum + your high-quality synthetic HF dataset

`load_dataset("csebuetnlp/xlsum", "nepali")` loads only the Nepali config directly from the
official dataset builder — no manual file-listing, no risk of pulling in other languages. Every
row here is guaranteed to be real Nepali news + summary pairs.

The synthetic dataset is now `iwasbinod/nep_summarization_1k_data-highquality` — the small,
curated, high-quality replacement for the old 5k low-quality set. It's loaded with the same
robust key-detection as before (works regardless of the exact column names in the new dataset),
then mixed into the training pool with a **hard no-duplication cap**.

In [4]:
from collections import Counter

DEVANAGARI_RE = re.compile(r"[\u0900-\u097F]")

# --- 2a. Real Nepali news data: load ONLY the 'nepali' config, nothing else ---
xlsum_raw = load_dataset("csebuetnlp/xlsum", "nepali", trust_remote_code=True)
print("XLSum (nepali config only) splits:", {k: len(v) for k, v in xlsum_raw.items()})

def standardize_xlsum(ds):
    keep = ds.remove_columns([c for c in ds.column_names if c not in ("text", "summary")])
    return keep.filter(lambda x: bool(x["text"]) and bool(x["summary"]) and len(x["text"]) > 50 and len(x["summary"]) > 10)

xlsum_data = DatasetDict({split: standardize_xlsum(ds) for split, ds in xlsum_raw.items()})
print("XLSum (Nepali-only) sizes after cleaning:", {k: len(v) for k, v in xlsum_data.items()})

sample = xlsum_data["train"].shuffle(seed=3407).select(range(min(200, len(xlsum_data["train"]))))
devanagari_rate = sum(1 for r in sample if DEVANAGARI_RE.search(r["text"])) / len(sample)
print(f"Devanagari coverage on 200-row sample: {devanagari_rate:.1%} (should be ~100%)")

# --- 2b. Your HIGH-QUALITY synthetic Nepali summarization dataset (v6: swapped in) ---
SYNTHETIC_DATASET_ID = "iwasbinod/nep_summarization_1k_data-highquality"

TEXT_KEYS = ["text", "article", "content", "document", "input", "body", "news", "prompt", "source_text"]
SUMMARY_KEYS = ["summary", "summaries", "target", "label", "output", "answer", "highlights", "completion", "response", "abstract"]

def first_nonempty_string(value):
    if value is None:
        return None
    if isinstance(value, str):
        value = value.strip()
        return value if value else None
    if isinstance(value, (int, float, bool)):
        return str(value).strip()
    return None

def extract_text_summary(example):
    text = None
    summary = None
    for key in TEXT_KEYS:
        if key in example:
            text = first_nonempty_string(example.get(key))
            if text:
                break
    for key in SUMMARY_KEYS:
        if key in example:
            summary = first_nonempty_string(example.get(key))
            if summary:
                break
    return text, summary

def standardize_split(dataset):
    def _map(example):
        text, summary = extract_text_summary(example)
        return {"text": text, "summary": summary}
    mapped = dataset.map(_map, remove_columns=dataset.column_names)
    mapped = mapped.filter(lambda x: bool(x["text"]) and bool(x["summary"]) and len(x["text"]) > 50 and len(x["summary"]) > 10)
    return mapped

supplemental_raw = load_dataset(SYNTHETIC_DATASET_ID)
supplemental_data = DatasetDict()
for split_name, split_ds in supplemental_raw.items():
    supplemental_data[split_name] = standardize_split(split_ds)

print(f"Supplemental (synthetic, {SYNTHETIC_DATASET_ID}) sizes BEFORE dedup:",
      {k: len(v) for k, v in supplemental_data.items()})

# ============================================================
# Dedup safety net — kept from v5 even though this set is curated/high-quality.
# ============================================================
# A curated 1k set is much less likely to contain templated duplicates than the old
# 5k low-quality set, but this is a cheap, harmless safety net: if it removes ~0 rows,
# that's itself a good confirmation the new data is clean.
def dedupe_synthetic_pool(dataset_dict):
    all_rows = concatenate_datasets(list(dataset_dict.values()))
    seen_pairs = set()
    seen_summaries = {}
    keep_idx = []
    for i, row in enumerate(all_rows):
        key = (row["text"].strip(), row["summary"].strip())
        summ_key = row["summary"].strip()
        if key in seen_pairs:
            continue
        seen_summaries[summ_key] = seen_summaries.get(summ_key, 0) + 1
        if seen_summaries[summ_key] > 2:
            continue
        seen_pairs.add(key)
        keep_idx.append(i)
    deduped = all_rows.select(keep_idx)
    n_removed = len(all_rows) - len(deduped)
    print(f"Synthetic dedup: removed {n_removed} / {len(all_rows)} rows "
          f"({n_removed / max(len(all_rows), 1):.1%}) as exact/near-duplicate rows")
    return deduped

supplemental_pool = dedupe_synthetic_pool(supplemental_data)

# Clean, leak-free train/validation split on the DEDUPED pool
split = supplemental_pool.train_test_split(test_size=0.10, seed=3407)
supplemental_data = DatasetDict({"train": split["train"], "validation": split["test"]})
print("Supplemental (synthetic) sizes AFTER dedup + re-split:", {k: len(v) for k, v in supplemental_data.items()})

# --- 2c. Build a weighted TRAINING pool (mix both sources) ---
# ============================================================
# v6 fix — GUARANTEED ZERO DUPLICATION for the small high-quality synthetic set
# ============================================================
# The v5 mixing function computed each source's quota as a share of the real-data pool size
# and only avoided duplication *if the quota happened to fit*. With ~5k real training rows and
# a 15% synthetic share, that's a ~750-row quota — usually fine against a 5k-row synthetic pool,
# but risky against a 1k-row high-quality one: if the quota ever exceeds the unique pool size,
# rows silently repeat within an epoch, which is exactly the memorization risk you're trying to
# avoid this round. This version hard-caps every source's quota at its own unique row count and
# pushes any shortfall onto sources that still have headroom (with the real XLSum-Nepali data
# absorbing the difference, since it's large). Net effect: the synthetic set is used AT MOST
# once per epoch, guaranteed, and the printed report below proves it every run.
SOURCE_MIX_WEIGHTS = {"xlsum": 0.85, "supplemental": 0.15}
TRAIN_TARGET_SIZE = len(xlsum_data["train"])  # scale to real data, don't inflate with synthetic

def sample_with_replacement(ds, target_len, seed):
    if ds is None or len(ds) == 0 or target_len <= 0:
        return None
    rng = random.Random(seed)
    if target_len <= len(ds):
        idxs = rng.sample(range(len(ds)), target_len)
    else:
        idxs = [rng.randrange(len(ds)) for _ in range(target_len)]
    return ds.select(idxs).shuffle(seed=seed)

def build_weighted_pool(named_datasets, weights, total_target, seed=3407, allow_duplication=None):
    """
    Builds a mixed training pool with per-source quotas proportional to `weights`.
    allow_duplication: dict[name] -> bool (default False for every source). When a source is
    not allowed to duplicate, its quota is capped at its own unique row count and the shortfall
    is redistributed to the remaining source(s) that still have headroom.
    """
    available = {name: ds for name, ds in named_datasets.items() if ds is not None and len(ds) > 0}
    if not available:
        raise ValueError("No datasets available to build the training pool.")
    if allow_duplication is None:
        allow_duplication = {name: False for name in available}

    weight_sum = sum(weights.get(name, 0.0) for name in available)
    quotas = {name: max(1, int(round(total_target * weights.get(name, 0.0) / weight_sum))) for name in available}

    # Cap sources that must not duplicate, and tally the shortfall this creates.
    shortfall = 0
    for name, ds in available.items():
        if not allow_duplication.get(name, False) and quotas[name] > len(ds):
            shortfall += quotas[name] - len(ds)
            quotas[name] = len(ds)

    # Redistribute the shortfall onto whichever source(s) still have headroom.
    if shortfall > 0:
        headroom_names = [n for n in available if quotas[n] < len(available[n]) or allow_duplication.get(n, False)]
        i = 0
        safety = shortfall * 10 + 10  # avoid an infinite loop if nothing has headroom
        while shortfall > 0 and headroom_names and safety > 0:
            name = headroom_names[i % len(headroom_names)]
            ds_size = len(available[name])
            if quotas[name] < ds_size or allow_duplication.get(name, False):
                quotas[name] += 1
                shortfall -= 1
            else:
                headroom_names.remove(name)
                if not headroom_names:
                    break
                continue
            i += 1
            safety -= 1
        if shortfall > 0:
            print(f"  NOTE: {shortfall} rows could not be placed without duplication anywhere; "
                  f"final training pool is smaller than TRAIN_TARGET_SIZE by that amount.")

    parts = []
    for idx, (name, ds) in enumerate(available.items()):
        part = sample_with_replacement(ds, quotas[name], seed + 97 * (idx + 1))
        if part is not None:
            part = part.add_column("source", [name] * len(part))
            parts.append(part)
    pool = concatenate_datasets(parts).shuffle(seed=seed)
    for name, q in quotas.items():
        dup_flag = "WITH duplication (unexpected!)" if q > len(available[name]) else "no duplication"
        print(f"  {name:14s}: {q} rows requested ({dup_flag}, source has {len(available[name])} unique rows)")
    return pool

print("Weighted training pool composition (synthetic capped to its own size — zero duplication):")
train_pool = build_weighted_pool(
    {"xlsum": xlsum_data["train"], "supplemental": supplemental_data.get("train")},
    SOURCE_MIX_WEIGHTS, TRAIN_TARGET_SIZE, seed=3407,
    allow_duplication={"xlsum": False, "supplemental": False},
)
print("Weighted training pool size:", len(train_pool))

# --- 2d. TWO SEPARATE held-out eval sets — kept apart on purpose ---
# real_eval: genuine Nepali news the model has never seen a template of -> tests true generalization
# synth_eval: your high-quality synthetic set's held-out slice -> tests fit to that style
N_EVAL = 60
xlsum_val_shuffled = xlsum_data["validation"].shuffle(seed=42)
real_eval = xlsum_val_shuffled.select(range(min(N_EVAL, len(xlsum_val_shuffled))))
# A SEPARATE slice (non-overlapping with real_eval) used for in-training validation,
# so the checkpoint-selection signal is never the same rows used for the final report.
real_val_for_training = xlsum_val_shuffled.select(
    range(min(N_EVAL, len(xlsum_val_shuffled)), min(N_EVAL + 60, len(xlsum_val_shuffled)))
)
synth_eval = supplemental_data["validation"].shuffle(seed=42).select(range(min(32, len(supplemental_data["validation"]))))
print(f"Real-world (XLSum Nepali) FINAL REPORT eval rows: {len(real_eval)}")
print(f"Real-world (XLSum Nepali) IN-TRAINING validation rows: {len(real_val_for_training)}")
print(f"Synthetic-style eval rows: {len(synth_eval)}")


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

XLSum (nepali config only) splits: {'train': 5808, 'test': 725, 'validation': 725}


Filter:   0%|          | 0/5808 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Supplemental (synthetic, iwasbinod/nep_summarization_1k_data-highquality) sizes BEFORE dedup: {'train': 1000}
Synthetic dedup: removed 0 / 1000 rows (0.0%) as exact/near-duplicate rows
Supplemental (synthetic) sizes AFTER dedup + re-split: {'train': 900, 'validation': 100}
Weighted training pool composition (synthetic capped to its own size — zero duplication):


Flattening the indices:   0%|          | 0/4937 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/871 [00:00<?, ? examples/s]

  xlsum         : 4937 rows requested (no duplication, source has 5808 unique rows)
  supplemental  : 871 rows requested (no duplication, source has 900 unique rows)
Weighted training pool size: 5808
Real-world (XLSum Nepali) FINAL REPORT eval rows: 60
Real-world (XLSum Nepali) IN-TRAINING validation rows: 60
Synthetic-style eval rows: 32


## 3. Load base model + attach QLoRA adapters

LoRA rank/alpha/dropout unchanged from the prior run (`r=16 / alpha=32 / dropout=0.05`) so
results are comparable to what you saw before.

In [5]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# lora_dropout raised 0.05 -> 0.1: extra regularization since the earlier run
# showed memorization symptoms (perfect synthetic-eval score). A slightly
# higher dropout makes it harder for the adapter to memorize exact template
# phrasing while barely affecting capacity at r=16.
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.1,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

terminators = [tokenizer.eos_token_id]
try:
    terminators.append(tokenizer.convert_tokens_to_ids("<|eot_id|>"))
except Exception:
    pass
terminators = [t for t in terminators if t is not None]
print("Stop tokens selected for inference safety:", terminators)


==((====))==  Unsloth 2026.7.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.6 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Stop tokens selected for inference safety: [128009, 128009]


## 4. Format prompts + length-filter the training split

In [6]:
SYSTEM_PROMPT = "तपाईं एक विशेषज्ञ नेपाली समाचार सारांशकर्ता हुनुहुन्छ।"

def format_prompt(text, summary=None):
    user_msg = (
        "दिइएको समाचारलाई ३-४ वाक्यमा छोटो, स्पष्ट र पूर्ण नेपालीमा सारांश गर्नुहोस्। "
        "मुख्य तथ्य, नाम, संख्या, र परिणामहरू समेट्नुहोस्। अंग्रेजी शब्द अनावश्यक रूपमा नराख्नुहोस्.\n\n"
        f"समाचार:\n{text}\n\nसारांश:"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    if summary is not None:
        messages.append({"role": "assistant", "content": summary})
    return messages

def format_training_example(batch):
    texts = [
        tokenizer.apply_chat_template(format_prompt(t, s), tokenize=False, add_generation_prompt=False)
        for t, s in zip(batch["text"], batch["summary"])
    ]
    return {"text": texts}

def within_length_limit(batch):
    lengths = [len(ids) for ids in tokenizer(batch["text"], add_special_tokens=False)["input_ids"]]
    return [l <= max_seq_length for l in lengths]

# --- Training set (real + synthetic mix, built in the data-mixing cell) ---
formatted_train = train_pool.map(format_training_example, batched=True, remove_columns=[c for c in train_pool.column_names if c != "source"])
formatted_train = formatted_train.filter(within_length_limit, batched=True, batch_size=64)
print(f"Dataset finalized for training loop: {len(formatted_train)} items.")
print("Source composition of final training set:", Counter(formatted_train["source"]))

# ============================================================
# CRITICAL FIX #3 — an in-training validation set made of ONLY real data
# ============================================================
# Without this, the trainer had no signal telling it when it started
# drifting away from real-world summarization quality and toward the
# synthetic template's shape. This is what earlier let training run
# blindly to a fixed max_steps=240 regardless of whether that was the
# best checkpoint for real-world generalization.
def format_eval_example(batch):
    texts = [
        tokenizer.apply_chat_template(format_prompt(t, s), tokenize=False, add_generation_prompt=False)
        for t, s in zip(batch["text"], batch["summary"])
    ]
    return {"text": texts}

formatted_val = real_val_for_training.map(format_eval_example, batched=True, remove_columns=real_val_for_training.column_names)
formatted_val = formatted_val.filter(within_length_limit, batched=True, batch_size=64)
print(f"In-training REAL validation set: {len(formatted_val)} items.")


Map:   0%|          | 0/5808 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5808 [00:00<?, ? examples/s]

Dataset finalized for training loop: 4465 items.
Source composition of final training set: Counter({'xlsum': 3594, 'supplemental': 871})


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Filter:   0%|          | 0/60 [00:00<?, ? examples/s]

In-training REAL validation set: 40 items.


## 5. Evaluation helpers

One evaluation function, used consistently everywhere (no more default-tokenizer ROUGE bug —
this always uses whitespace tokenization, which handles Devanagari correctly). Reports ROUGE,
chrF++, BERTScore, length ratio, empty/Devanagari-coverage checks, a 0-100 composite score, and
how much the output just copies the source (lower = more abstractive, less parroting).

In [7]:
rouge = hf_evaluate.load("rouge")

def whitespace_tokenizer(text):
    return text.split()

def tokenise_for_overlap(text):
    return [tok for tok in re.findall(r"[\u0900-\u097F]+|[A-Za-z0-9]+", text.lower()) if tok.strip()]

def jaccard_similarity(a, b):
    a_set, b_set = set(a), set(b)
    if not a_set and not b_set:
        return 1.0
    if not a_set or not b_set:
        return 0.0
    return len(a_set & b_set) / len(a_set | b_set)

# v6 fix: 200 -> 320. Devanagari commonly tokenizes into more sub-word pieces than English for
# the same semantic content, so 200 tokens was cutting some summaries off mid-sentence before
# they reached a natural stop token — this is the same failure mode that led to summarization
# being deprecated for LLaMA/Mistral in your other runs. 320 gives real headroom for a
# 3-4 sentence Nepali summary to finish naturally and still hit its EOS token.
def generate_summary(text, max_new_tokens=320):
    messages = format_prompt(text)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_seq_length, padding=False).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            top_p=0.90,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=terminators,
        )
    return tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def evaluate_all(preds, refs, sources=None):
    rouge_scores = rouge.compute(predictions=preds, references=refs, tokenizer=whitespace_tokenizer)
    chrf = sacrebleu.corpus_chrf(preds, [refs], word_order=2)
    P, R, F1 = bert_score(preds, refs, lang="ne", model_type="bert-base-multilingual-cased", verbose=False)

    pred_lengths = [len(p.split()) for p in preds]
    ref_lengths = [len(r.split()) for r in refs]
    empty_rate = sum(1 for p in preds if not p.strip()) / max(len(preds), 1)
    devanagari_rate = sum(1 for p in preds if DEVANAGARI_RE.search(p)) / max(len(preds), 1)
    avg_len_ratio = (sum(pred_lengths) / max(sum(ref_lengths), 1)) if refs else 0.0

    # Cheap truncation-sanity check: a summary that doesn't end in Devanagari danda (।),
    # a standard '.', '?', or '!' is a strong signal generation was cut off mid-sentence.
    SENTENCE_END_CHARS = ("।", ".", "?", "!", '"', "'")
    truncated_rate = sum(
        1 for p in preds if p.strip() and not p.strip().endswith(SENTENCE_END_CHARS)
    ) / max(len(preds), 1)

    copy_overlap = None
    if sources is not None:
        overlaps = [jaccard_similarity(tokenise_for_overlap(src), tokenise_for_overlap(pred)) for src, pred in zip(sources, preds)]
        copy_overlap = sum(overlaps) / max(len(overlaps), 1)

    composite = 0.35 * (rouge_scores["rougeL"] * 100.0) + 0.35 * chrf.score + 0.30 * (F1.mean().item() * 100.0)

    out = {
        "rouge1": rouge_scores["rouge1"], "rouge2": rouge_scores["rouge2"], "rougeL": rouge_scores["rougeL"],
        "chrF++": chrf.score, "bertscore_f1": F1.mean().item(),
        "avg_pred_len": sum(pred_lengths) / max(len(pred_lengths), 1),
        "avg_ref_len": sum(ref_lengths) / max(len(ref_lengths), 1),
        "avg_len_ratio": avg_len_ratio, "empty_rate": empty_rate, "devanagari_rate": devanagari_rate,
        "likely_truncated_rate": truncated_rate,
        "composite_0_100": composite,
    }
    if copy_overlap is not None:
        out["source_copy_jaccard"] = copy_overlap
    return out

def print_validation_report(name, metrics):
    print(f"\n{name}")
    print("-" * len(name))
    for k, v in metrics.items():
        print(f"{k:<22}: {v:.4f}" if isinstance(v, float) else f"{k:<22}: {v}")
    if metrics.get("likely_truncated_rate", 0.0) > 0.10:
        print("  WARNING: >10% of outputs don't end on sentence-final punctuation — likely "
              "mid-sentence truncation. Consider raising max_new_tokens further.")


## 6. Baseline (zero-shot) evaluation — real-world AND synthetic, reported separately

In [8]:
FastLanguageModel.for_inference(model)

real_sources = [ex["text"] for ex in real_eval]
real_refs = [ex["summary"] for ex in real_eval]
synth_sources = [ex["text"] for ex in synth_eval]
synth_refs = [ex["summary"] for ex in synth_eval]

print("Generating base-model predictions on REAL XLSum-Nepali eval set...")
t0 = time.time()
base_preds_real = [generate_summary(t) for t in real_sources]
print(f"Done in {(time.time()-t0)/60:.1f} min")

print("Generating base-model predictions on SYNTHETIC eval set...")
t0 = time.time()
base_preds_synth = [generate_summary(t) for t in synth_sources]
print(f"Done in {(time.time()-t0)/60:.1f} min")

base_metrics_real = evaluate_all(base_preds_real, real_refs, sources=real_sources)
base_metrics_synth = evaluate_all(base_preds_synth, synth_refs, sources=synth_sources)

print_validation_report("BASE MODEL — REAL XLSum-Nepali", base_metrics_real)
print_validation_report("BASE MODEL — SYNTHETIC-STYLE", base_metrics_synth)


Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating base-model predictions on REAL XLSum-Nepali eval set...


Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Done in 11.1 min
Generating base-model predictions on SYNTHETIC eval set...


Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



BASE MODEL — REAL XLSum-Nepali
------------------------------
rouge1                : 0.1147
rouge2                : 0.0247
rougeL                : 0.0904
chrF++                : 22.9222
bertscore_f1          : 0.6755
avg_pred_len          : 48.8833
avg_ref_len           : 18.7833
avg_len_ratio         : 2.6025
empty_rate            : 0.0000
devanagari_rate       : 1.0000
likely_truncated_rate : 0.2833
composite_0_100       : 31.4498
source_copy_jaccard   : 0.1255

BASE MODEL — SYNTHETIC-STYLE
----------------------------
rouge1                : 0.4101
rouge2                : 0.2452
rougeL                : 0.3449
chrF++                : 48.1756
bertscore_f1          : 0.8056
avg_pred_len          : 30.9062
avg_ref_len           : 22.1250
avg_len_ratio         : 1.3969
empty_rate            : 0.0000
devanagari_rate       : 1.0000
likely_truncated_rate : 0.0938
composite_0_100       : 53.0991
source_copy_jaccard   : 0.3271


## 7. Train the adapter

In [9]:
import gc
import torch
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
from transformers import EarlyStoppingCallback

# v6: renamed so this never collides with the v4/v5 adapters (or any earlier overfit adapter).
# Hugging Face repo IDs don't allow '+', so "real+syn-highquality" is rendered as
# "real-syn-highquality".
ADAPTER_REPO_SLUG = "summarization-llama3.2-3b-real-syn-highquality"
LOCAL_ADAPTER_DIR = "nepali_summarization_llama_lora_model_real_syn_highquality"

# ============================================================
# Anti-overfitting training setup (carried over from v5, unchanged)
# ============================================================
#   - max_steps caps total training to a predictable, Kaggle-session-safe budget instead of
#     scaling uncontrolled with epochs.
#   - prediction_loss_only=True + eval_accumulation_steps=1 keep the periodic real-data eval
#     pass cheap (scalar loss only, no full-vocab logits held on GPU).
#   - eval_steps/save_steps=70, load_best_model_at_end + EarlyStoppingCallback: the checkpoint
#     that's actually kept is whichever one had the LOWEST loss on held-out REAL XLSum-Nepali
#     data, not whatever the last step happened to be — this is the mechanism that gives you the
#     best available shot at "always better than base" without hand-tuning stopping points.
#   - learning_rate=1e-4 + lora_dropout=0.1 (see LoRA cell above) + weight_decay=0.05: gentle
#     updates and extra regularization so a few hundred steps over a small high-quality synthetic
#     slice can't push the adapter into memorizing surface patterns.
training_args = SFTConfig(
    output_dir="outputs-v6",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_ratio=0.05,
    max_steps=350,
    learning_rate=1e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.05,
    lr_scheduler_type="linear",
    seed=3407,
    dataset_text_field="text",
    eval_strategy="steps",
    eval_steps=70,
    save_strategy="steps",
    save_steps=70,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    prediction_loss_only=True,
    eval_accumulation_steps=1,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_train,
    eval_dataset=formatted_val,
    dataset_num_proc=2,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

gc.collect()
torch.cuda.empty_cache()

trainer_stats = trainer.train()

print(f"\nBest checkpoint (lowest REAL-data eval loss) was auto-restored: "
      f"{trainer.state.best_model_checkpoint}")
print(f"Best eval loss: {trainer.state.best_metric:.4f}")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/4465 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/40 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/4465 [00:00<?, ? examples/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,465 | Num Epochs = 1 | Total steps = 350
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
70,1.113270,1.246802
140,1.072206,1.170372
210,1.077841,1.118067
280,1.110875,1.091758
350,0.955676,1.084956


Unsloth: Restored added_tokens_decoder metadata in outputs-v6/checkpoint-70/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs-v6/checkpoint-140/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs-v6/checkpoint-210/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs-v6/checkpoint-280/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs-v6/checkpoint-350/tokenizer_config.json.



Best checkpoint (lowest REAL-data eval loss) was auto-restored: outputs-v6/checkpoint-350
Best eval loss: 1.0850


## 8. Post-training evaluation — same two eval sets, same metrics

In [10]:
FastLanguageModel.for_inference(model)

print("Generating fine-tuned predictions on REAL XLSum-Nepali eval set...")
t0 = time.time()
ft_preds_real = [generate_summary(t) for t in real_sources]
print(f"Done in {(time.time()-t0)/60:.1f} min")

print("Generating fine-tuned predictions on SYNTHETIC eval set...")
t0 = time.time()
ft_preds_synth = [generate_summary(t) for t in synth_sources]
print(f"Done in {(time.time()-t0)/60:.1f} min")

ft_metrics_real = evaluate_all(ft_preds_real, real_refs, sources=real_sources)
ft_metrics_synth = evaluate_all(ft_preds_synth, synth_refs, sources=synth_sources)

print_validation_report("FINE-TUNED — REAL XLSum-Nepali", ft_metrics_real)
print_validation_report("FINE-TUNED — SYNTHETIC-STYLE", ft_metrics_synth)

print("\n=== DELTA: composite score (0-100) ===")
print(f"Real XLSum-Nepali : {base_metrics_real['composite_0_100']:.2f} -> {ft_metrics_real['composite_0_100']:.2f}  "
      f"(delta {ft_metrics_real['composite_0_100'] - base_metrics_real['composite_0_100']:+.2f})")
print(f"Synthetic-style   : {base_metrics_synth['composite_0_100']:.2f} -> {ft_metrics_synth['composite_0_100']:.2f}  "
      f"(delta {ft_metrics_synth['composite_0_100'] - base_metrics_synth['composite_0_100']:+.2f})")
print("\nThe REAL row is the one that tells you whether this generalizes.")
print("A big jump on SYNTHETIC alone with a flat/small REAL delta means the adapter is")
print("mostly memorizing your template family rather than learning general summarization.")

# ============================================================
# Factual-grounding check — catches "style transfer without substance"
# ============================================================
# ROUGE/chrF/BERTScore alone would not have caught the earlier failure mode:
# the fine-tuned model got shorter, more fluent-looking summaries on REAL
# data, but source_copy_jaccard (how many source words show up in the
# summary) actually fell, meaning it was inventing generic phrasing instead
# of extracting facts. Track this explicitly every run.
real_jaccard_delta = ft_metrics_real["source_copy_jaccard"] - base_metrics_real["source_copy_jaccard"]
print("\n=== FACTUAL GROUNDING CHECK (source_copy_jaccard, REAL data) ===")
print(f"Base       : {base_metrics_real['source_copy_jaccard']:.4f}")
print(f"Fine-tuned : {ft_metrics_real['source_copy_jaccard']:.4f}  (delta {real_jaccard_delta:+.4f})")
if real_jaccard_delta < -0.02:
    print("WARNING: grounding in the source article got WORSE after fine-tuning.")
    print("This usually means the model is producing fluent but generic/hallucinated")
    print("summaries rather than genuinely extracting facts — inspect the qualitative")
    print("samples below closely before trusting the ROUGE improvement.")
elif real_jaccard_delta > 0.02:
    print("Grounding improved — fine-tuned summaries reference the source article more.")
else:
    print("Grounding roughly unchanged.")


Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating fine-tuned predictions on REAL XLSum-Nepali eval set...


Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Done in 4.5 min
Generating fine-tuned predictions on SYNTHETIC eval set...


Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



FINE-TUNED — REAL XLSum-Nepali
------------------------------
rouge1                : 0.1683
rouge2                : 0.0525
rougeL                : 0.1591
chrF++                : 23.4638
bertscore_f1          : 0.7363
avg_pred_len          : 14.7500
avg_ref_len           : 18.7833
avg_len_ratio         : 0.7853
empty_rate            : 0.0000
devanagari_rate       : 1.0000
likely_truncated_rate : 0.0000
composite_0_100       : 35.8687
source_copy_jaccard   : 0.0497

FINE-TUNED — SYNTHETIC-STYLE
----------------------------
rouge1                : 0.6791
rouge2                : 0.5271
rougeL                : 0.6514
chrF++                : 69.9891
bertscore_f1          : 0.8989
avg_pred_len          : 22.7500
avg_ref_len           : 22.1250
avg_len_ratio         : 1.0282
empty_rate            : 0.0000
devanagari_rate       : 1.0000
likely_truncated_rate : 0.0000
composite_0_100       : 74.2644
source_copy_jaccard   : 0.3781

=== DELTA: composite score (0-100) ===
Real XLSum-Nepali : 31.4

## 9. Sample side-by-side comparisons (real-world set)

In [11]:
N_SHOW = 3
print("=== REAL XLSum-Nepali samples ===")
for i in range(min(N_SHOW, len(real_eval))):
    print(f"\n--- Example {i+1} ---")
    print("Source (excerpt):", real_sources[i][:300], "...")
    print("Gold summary     :", real_refs[i])
    print("Base model       :", base_preds_real[i])
    print("Fine-tuned       :", ft_preds_real[i])

print("\n\n=== SYNTHETIC-style samples ===")
for i in range(min(N_SHOW, len(synth_eval))):
    print(f"\n--- Example {i+1} ---")
    print("Source (excerpt):", synth_sources[i][:300], "...")
    print("Gold summary     :", synth_refs[i])
    print("Base model       :", base_preds_synth[i])
    print("Fine-tuned       :", ft_preds_synth[i])


=== REAL XLSum-Nepali samples ===

--- Example 1 ---
Source (excerpt): मङ्गलवार उनलाई भेट्न गएका पूर्वप्रधानन्यायाधीश कल्याण श्रेष्ठ भन्छन् - मेरो अन्तर्मनको आवाज सुनेर म डा. गोविन्द केसीलाई भेट्न गएको हुँ। हिजो (मङ्गलवार) बिहान ओछ्यानमै थिएँ। डा. केसीले अनशन गरेको २१ दिन भयो, उहाँको स्वास्थ चिन्ताजनक छ भन्ने समाचार सुनेँ। कुनै सम्वाद भएको छैन भन्ने सुनेपछि संवेदनशीलता ...
Gold summary     : अनशनरत डा. गोविन्द केसीको स्वास्थ्यस्थिति जटिल बन्दै गएको चिकित्सकहरूले बताइरहँदा समाजका विभिन्न तप्काका मानिसहरूले उनलाई भेटिरहेका छन्।
Base model       : ू कित्ता गरेर कुनै कुरा गरेर कुनै कुरा नभएको देख्छु।
Fine-tuned       : ू कित्ता निकै भिन्न रहेको छ।

--- Example 2 ---
Source (excerpt): हामी उनीहरूसँग ९ हजार वर्षभन्दा पहिलादेखि सँगै बसिरहेका छौँ। अधिकांश समय सुत्ने भएपनि उनीहरू विश्वकै सबैभन्दा लोकप्रिय घरपालुवा जन्तु पनि हुन्। तर तपाईँलाई यिनीहरूबारे कति जानकारी छ? उनीहरूका आश्चर्यमा पार्ने विशेषता यस्ता छन्। उनीहरूले नियमित रूपमा इन्टरनेटमा चर्चित हुन्छन्। उनीहरुले मानवलाई खासै म ...
Gold s

## 10. Push adapter to Hugging Face Hub

Adapter repo slug: `iwasbinod/summarization-llama3.2-3b-real-syn-highquality`.

**Before you run this cell**, scroll up to the "DELTA: composite score" and "FACTUAL GROUNDING
CHECK" printouts from section 8. If the REAL-XLSum-Nepali composite score went *down*, or the
factual grounding warning fired, the adapter did not clearly beat the base model on real data —
you may still want to push it for record-keeping, but don't treat it as a final answer without
looking at the qualitative samples in section 9 first.

In [12]:
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in via Kaggle Secret HF_TOKEN.")
except Exception:
    print("No Kaggle Secret named HF_TOKEN found - add one via Add-ons > Secrets to skip this prompt next time.")
    from huggingface_hub import notebook_login
    notebook_login()

hf_username = "iwasbinod"
repo_name = ADAPTER_REPO_SLUG
full_repo_id = f"{hf_username}/{repo_name}"

# Non-blocking sanity check before pushing: did the fine-tuned model actually beat the base
# model on REAL data? This won't stop the push (you may still want the adapter saved), but it
# makes sure you don't miss a regression.
real_delta = ft_metrics_real["composite_0_100"] - base_metrics_real["composite_0_100"]
if real_delta < 0:
    print(f"WARNING: REAL-data composite score REGRESSED by {real_delta:.2f} points vs. the base "
          f"model. Pushing anyway, but review the qualitative samples above before using this "
          f"adapter as your final result.")
else:
    print(f"REAL-data composite score improved by {real_delta:+.2f} points vs. the base model. Good sign.")

model.save_pretrained(LOCAL_ADAPTER_DIR)
tokenizer.save_pretrained(LOCAL_ADAPTER_DIR)
print(f"Saved locally to {LOCAL_ADAPTER_DIR}/ - this part cannot fail due to network issues.")

model.push_to_hub(full_repo_id)
tokenizer.push_to_hub(full_repo_id)
print(f"Adapter pushed to https://huggingface.co/{full_repo_id}")


Logged in via Kaggle Secret HF_TOKEN.
REAL-data composite score improved by +4.42 points vs. the base model. Good sign.


Unsloth: Restored added_tokens_decoder metadata in nepali_summarization_llama_lora_model_real_syn_highquality/tokenizer_config.json.


Saved locally to nepali_summarization_llama_lora_model_real_syn_highquality/ - this part cannot fail due to network issues.


README.md:   0%|          | 0.00/567 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmps3pqf0lt/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Adapter pushed to https://huggingface.co/iwasbinod/summarization-llama3.2-3b-real-syn-highquality
